# Run Remaining Training Models (Local or Kaggle)\n
\n
This notebook runs all remaining training phases after Component 1 and Component 4 are pretrained.

In [ ]:
import os\n
import subprocess\n
from pathlib import Path\n
\n
REPO = Path.cwd()\n
print('Repo:', REPO)

In [ ]:
def run(cmd):\n
    print('\n>>>', cmd)\n
    result = subprocess.run(cmd, shell=True, text=True)\n
    if result.returncode != 0:\n
        raise RuntimeError(f'Command failed with code {result.returncode}: {cmd}')

## Config\n
Set these before running.

In [ ]:
CONFIG_MOE = 'configs/moe.yaml'\n
CONFIG_C2 = 'configs/component2_txv.yaml'\n
CONFIG_PATHS = 'configs/paths.yaml'\n
CONFIG_C1 = 'configs/component1_dann.yaml'\n
CACHE_DIR = 'outputs/moe_runs/embedding_cache'\n
MODE = 'full'  # smoke | short | full

## 1) Train Component 2 Routing Head

In [ ]:
run(f'python -m src.training.train_component2_txv --config {CONFIG_C2} --paths {CONFIG_PATHS} --component1_config {CONFIG_C1}')

## 2) Build MoE Cache

In [ ]:
run(f'python scripts/cache_moe_embeddings.py --config {CONFIG_MOE} --paths {CONFIG_PATHS} --output {CACHE_DIR}')

## 3) Expert Pretraining (Component 5)

In [ ]:
run(f'python -m src.training.train_experts --config {CONFIG_MOE} --cache-dir {CACHE_DIR} --all')

## 4) Joint MoE Training (Components 3 + 5 + 6)

In [ ]:
run(f'python -m src.training.train_moe_joint --config {CONFIG_MOE} --cache-dir {CACHE_DIR}')

## 5) Boundary Critic Training (Component 7)

In [ ]:
run(f'python -m src.training.train_boundary_critic --config {CONFIG_MOE} --cache-dir {CACHE_DIR}')

## 6) Quick Checkpoint Summary

In [ ]:
for p in sorted(Path('outputs/moe_runs').glob('*')):\n
    print(p)

## 7) Optional Inference Sanity Test

In [ ]:
# Replace image path and dataset id before running\n
# run('python -m src.app.infer --image path/to/image.png --dataset montgomery --outdir outputs/demo_after_training --config configs/moe.yaml')